# 02 — Model Training Walkthrough
### Multimodal Disease Detection Framework

This notebook demonstrates building and training the full multimodal model step by step.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from models.multimodal_model import MultimodalDiseaseDetector
from models.cross_modal_attention import CrossModalAttention
from utils.losses import MultimodalLoss
from utils.metrics import compute_metrics, print_metrics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Build the Model

In [ ]:
model = MultimodalDiseaseDetector(
    num_classes=1,
    latent_dim=256,
    img_backbone='resnet50',
    ecg_input_dim=1,
    ehr_input_dim=64,
    modality_dropout_prob=0.3,
    num_attention_heads=4,
    pretrained_cnn=True
).to(device)

param_counts = model.count_parameters()
print('\nParameter Counts:')
for k, v in param_counts.items():
    print(f'  {k:<20}: {v:>12,}')

## 2. Loss Function (Eq. 9 from Paper)

In [ ]:
# L = L_classification + lambda * L_regularization + mu * L_alignment
loss_fn = MultimodalLoss(
    lambda_reg=1e-4,    # λ
    mu_align=0.1,       # μ
    num_classes=1
)

# Test with random batch
logits = torch.randn(8, 1)
labels = torch.randint(0, 2, (8,)).float()
feats  = [torch.randn(8, 256) for _ in range(3)]
losses = loss_fn(logits, labels, modality_features=feats)

print('Loss Components:')
for k, v in losses.items():
    print(f'  {k:<18}: {v.item():.4f}')

## 3. Cross-Modal Attention — Visualization

In [ ]:
# Simulate attention weights for different clinical scenarios
np.random.seed(42)

# Scenario 1: Clear imaging + good EHR (imaging dominates)
# Scenario 2: Blurry imaging + strong ECG anomaly
# Scenario 3: All modalities balanced
scenarios = {
    'Clear Imaging': [0.55, 0.20, 0.25],
    'ECG Anomaly':   [0.15, 0.60, 0.25],
    'Balanced':      [0.35, 0.33, 0.32],
}

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(3)
width = 0.25
modalities = ['Imaging (CNN)', 'ECG (LSTM)', 'EHR (MLP)']
colors = ['#2196F3', '#F44336', '#4CAF50']

for i, (scenario, weights) in enumerate(scenarios.items()):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, weights, width, label=scenario, edgecolor='black', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(modalities, fontsize=11)
ax.set_ylabel('Attention Weight (α)', fontsize=12)
ax.set_title('Cross-Modal Attention Weights Under Different Clinical Scenarios', fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/attention_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Forward Pass — All Modalities

In [ ]:
model.eval()
with torch.no_grad():
    B = 4
    images = torch.randn(B, 3, 224, 224).to(device)
    ecg    = torch.randn(B, 1500, 1).to(device)
    ehr    = torch.randn(B, 64).to(device)

    out = model(images=images, ecg=ecg, ehr=ehr)

print('Output keys:', list(out.keys()))
print(f"Probabilities: {out['probabilities'].squeeze().cpu().numpy().round(3)}")
print(f"Attention weights (img | ecg | ehr):")
for i, w in enumerate(out['attention_weights'].cpu().numpy()):
    print(f"  Sample {i+1}: {w[0]:.3f} | {w[1]:.3f} | {w[2]:.3f}")

## 5. Modality Robustness Test (Table 7 from Paper)

In [ ]:
# Simulate performance under missing modalities (from paper Table 7)
robustness_results = {
    'All Modalities':    {'AUC': 0.94, 'Precision': 0.91, 'Recall': 0.88},
    'Missing Imaging':   {'AUC': 0.88, 'Precision': 0.85, 'Recall': 0.84},
    'Missing EHR':       {'AUC': 0.89, 'Precision': 0.86, 'Recall': 0.85},
    'Missing Time-series': {'AUC': 0.90, 'Precision': 0.87, 'Recall': 0.86},
}

configs = list(robustness_results.keys())
metrics = ['AUC', 'Precision', 'Recall']
x = np.arange(len(configs))
width = 0.25
metric_colors = ['#2196F3', '#4CAF50', '#F44336']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, color) in enumerate(zip(metrics, metric_colors)):
    vals = [robustness_results[c][metric] for c in configs]
    ax.bar(x + (i-1)*width, vals, width, label=metric, color=color, edgecolor='black', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(configs, fontsize=10)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Robustness Under Missing Modalities (Table 7)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/robustness_test.png', dpi=150, bbox_inches='tight')
plt.show()